First we import stuff

In [ ]:
from pandapower.control import ConstControl

import pandapipes as pp

import numpy as np
import copy
import matplotlib.pyplot as plt

import tempfile

import pandas as pd
from pandapipes.timeseries import run_timeseries, init_default_outputwriter
from pandapower.timeseries import OutputWriter, DFData

We create a simple circular heating network consisting of a pump, two pipes and a heat consumer

In [ ]:
net = pp.create_empty_network(fluid="water")
# create junctions
j1 = pp.create_junction(net, pn_bar=1.05, tfluid_k=330, name="Junction 1")
j2 = pp.create_junction(net, pn_bar=1.05, tfluid_k=330, name="Junction 2")
j3 = pp.create_junction(net, pn_bar=1.05, tfluid_k=300, name="Junction 3")
j4 = pp.create_junction(net, pn_bar=1.05, tfluid_k=300, name="Junction 4")

# create junction elements
circ_pump = pp.create_circ_pump_const_pressure(net, return_junction=j4, flow_junction=j1, p_flow_bar=3, plift_bar=0.8,
                                               t_flow_k=350, name="Grid Connection")
consumer = pp.create_heat_consumer(net, from_junction=j2, to_junction=j3, qext_w=5000, controlled_mdot_kg_per_s=0.1)

print(net)

For the pipes we define sections, since those are necessary for transient calculations to simulate the pipe as a series of finite elements.

In [ ]:
# create branch elements
sections = 5
length = 0.5
pp.create_pipe_from_parameters(net, j1, j2, length, 75e-3, k_mm=.0472, sections=sections,
                               u_w_per_m2k=2, text_k=293)
pp.create_pipe_from_parameters(net, j3, j4, length, 75e-3, k_mm=.0472, sections=sections,
                               u_w_per_m2k=2, text_k=293)
print(net.pipe)

Next we define the input data for the timeseries calculation. In this case the temperature at the pump outlet.

In [ ]:
ds = DFData(pd.DataFrame({"t_k": [330] * 50 + [335] * 5+ [340] * 5+ [345] * 5+ [350] * 5+ [360] * 100}))
t_ctrl = ConstControl(net, element="circ_pump_pressure", variable="t_flow_k", element_index=0, profile_name="t_k", data_source=ds)

And the length of the timesteps we want to simulate.

In [ ]:
dt_simulation = 50

ow = OutputWriter(net, time_steps=None, log_variables=[
        ("res_junction", "t_k"),
        ("res_junction", "p_bar"),
        ("res_pipe", "v_mean_m_per_s"),
        ("res_pipe", "t_outlet_k")])

Then we run the simulation

In [ ]:
run_timeseries(net, continue_on_divergence=False, verbose=True,
                   mode="sequential", transient=True, dt=dt_simulation, iter=200)

Now we can extract the data

In [ ]:
res_T = ow.np_results["res_junction.t_k"]
res_T_df = pd.DataFrame(res_T)

We prepare plotting

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# res_T shape: (timesteps, 4) -> [flow_start, flow_end, return_start, return_end]
timesteps = np.arange(res_T.shape[0])

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(timesteps, res_T[:, 0], color="blue", linestyle="dashed", alpha=0.5, label="Flow Pump")
plt.plot(timesteps, res_T[:, 1], color="blue", linestyle="dotted", alpha=0.5, label="Flow Consumer")
plt.plot(timesteps, res_T[:, 2], color="red", linestyle="dashed", alpha=0.5, label="Return Consumer")
plt.plot(timesteps, res_T[:, 3], color="red", linestyle="dotted", alpha=0.5, label="Return Pump")

plt.title("Temperature Evolution Over Time", fontsize=14)
plt.xlabel("Time Step", fontsize=12)
plt.ylabel("Temperature [K]", fontsize=12)
plt.ylim(290, 370)
plt.grid(alpha=0.3)
plt.legend(loc="best")
plt.tight_layout()

plt.show(block=True)